# Packages

In [ ]:
import pandas as pd
from scipy.stats import ttest_ind, chi2_contingency, pearsonr
import scipy.stats
import os, sys
import matplotlib.pyplot as plt
import subprocess

In [ ]:
#Versions Used For This Data Analysis Script
print(f"panads: {pd.__version__}") #2.1.4
print(f"Platform System: {os.name}") #posix
print(f"Python: {sys.version}") #3.12.3 (v3.12.3:f6650f9ad7, Apr  9 2024, 08:18:48) [Clang 13.0.0 (clang-1300.0.29.30)]

# Dataset Selection

In [ ]:
version = 4

In [ ]:
#Load data
data_dir = f"../preprocessed-data/full-data-version-{version}/"

for data_file in os.listdir(data_dir):
    if "nback" in data_file:
        nback_data = pd.read_csv(data_dir+data_file)
        print(f"Processed '{data_file}' from '{data_dir}' as nback_data")
    elif "FITB" in data_file:
        FITB_data = pd.read_csv(data_dir+data_file)
        print(f"Processed '{data_file}' from '{data_dir}' as FITB_data")



Analyze the FITB code for accuracy

In [ ]:
data = FITB_data[FITB_data["problem_type"] == 'short']


for i in range(len(data["response_minus_initial"])):
    try:
        if "a = b" in data["response_minus_initial"][i]:
            display(data["response_minus_initial"][i])
    except:
        pass


# Analyze Data Using Student's T-Test

For the purpose of this prelim analysis, we will ignore session no. and just compare treatment types per task type

In [ ]:
#fuck - do literaure review: code comprension related to identifer names (have the found a relationship )
#Janet Siegmund - code complexity metrics and code comprehension -> number of variables is a key component here (morphology vs variable names)
coding_types={"short", "long"} 
FITB_types = {1,2,3}
results = {}

print("RESPONSE TIMES")
for coding_type in coding_types:
    cm_group = FITB_data.query(f"TMS_Type=='CM' & problem_type=='{coding_type}'")["duration"] # control region, response time
    wm_group = FITB_data.query(f"TMS_Type=='WM' & problem_type=='{coding_type}'")["duration"] # working memory, response time
    f_value, p_value = scipy.stats.f_oneway(cm_group, wm_group)
    results[coding_type] = {"f_value": f_value, "p_value": p_value, "is_significant": p_value < 0.05, "cm_lessthan_wm_mean": cm_group.mean() < wm_group.mean()}
    
for var_no in FITB_types:
    cm_group = FITB_data.query(f"TMS_Type=='CM' & identifer_no=={var_no}")["duration"]
    wm_group = FITB_data.query(f"TMS_Type=='WM' & identifer_no=={var_no}")["duration"]
    f_value, p_value = scipy.stats.f_oneway(cm_group, wm_group)
    results[str(var_no)+"-var"] = {"f_value": f_value, "p_value": p_value, "is_significant": p_value < 0.05, "cm_lessthan_wm_mean": cm_group.mean() < wm_group.mean()}
    
for nback, stats in results.items():
    print(f"{nback}: f-value = {stats['f_value']}, p-value = {stats['p_value']}, is_significant = {stats['is_significant']}, (Control RT < Treatment RT) mean = {stats['cm_lessthan_wm_mean']}")



In [ ]:
coding_types={"short", "long"}
FITB_types = {1,2,3}
results = {}
print("KEYPRESSING SPEED - key press per second")

#looking at deleting (maybe type faster and deleting more) -comapring to wenxin paper

for coding_type in coding_types:
    cm_group = FITB_data.query(f"TMS_Type=='CM' & problem_type=='{coding_type}'")["keypress_rate (keypresses per second)"]
    wm_group = FITB_data.query(f"TMS_Type=='WM' & problem_type=='{coding_type}'")["keypress_rate (keypresses per second)"]
    f_value, p_value = scipy.stats.f_oneway(cm_group, wm_group)
    results[coding_type] = {"f_value": f_value, "p_value": p_value, "is_significant": p_value < 0.05, "cm_lessthan_wm_mean": cm_group.mean() < wm_group.mean()}
    
for var_no in FITB_types:
    cm_group = FITB_data.query(f"TMS_Type=='CM' & identifer_no=={var_no}")["keypress_rate (keypresses per second)"]
    wm_group = FITB_data.query(f"TMS_Type=='WM' & identifer_no=={var_no}")["keypress_rate (keypresses per second)"]
    f_value, p_value = scipy.stats.f_oneway(cm_group, wm_group)
    results[str(var_no)+"-var"] = {"f_value": f_value, "p_value": p_value, "is_significant": p_value < 0.05, "cm_lessthan_wm_mean": cm_group.mean() < wm_group.mean()}
    
for nback, stats in results.items():
    print(f"{nback}: f-value = {stats['f_value']}, p-value = {stats['p_value']}, is_significant = {stats['is_significant']}, (Control KS < Treatment KS) mean = {stats['cm_lessthan_wm_mean']}")


n-back response types and accuracy via ttest

In [ ]:
results = {}

# Exclude rows where 'choose_nothing' is True
data = nback_data[nback_data["chose_nothing"] == False]
# (how many are there)

# Group data by n-back type
for nback_type, group in data.groupby("nback_type"):
    # Split data by TMS Type
    cm_group = group[group["TMS_Type"] == "CM"]["response time"]
    wm_group = group[group["TMS_Type"] == "WM"]["response time"]

    
    # Perform the t-test if both groups have data
    if len(cm_group) > 1 and len(wm_group) > 1:  # Ensure sufficient data
        t_stat, p_value = ttest_ind(cm_group, wm_group, equal_var=True)  # Student t-test
        results[nback_type] = {"t_stat": t_stat, "p_value": p_value, "is_significant": p_value < 0.05, "cm_lessthan_wm_mean": cm_group.mean() < wm_group.mean()}
    else:
        results[nback_type] = {"t_stat": None, "p_value": None, "is_significant": None, "cm_lessthan_wm_mean": None}

# Display results
for nback, stats in results.items():
    print(f"{nback}: t-stat = {stats['t_stat']}, p-value = {stats['p_value']}, is_significant = {stats['is_significant']}, (Control RT < Treatment RT) mean = {stats['cm_lessthan_wm_mean']}")

In [ ]:
# Filter out rows where chose_nothing is True
data = nback_data[nback_data["chose_nothing"] == False]

# Initialize results dictionary
results = {}

# Perform chi-squared test for each n-back type
for nback_type, group in data.groupby("nback_type"):
    # Create a contingency table
    contingency_table = pd.crosstab(group["TMS_Type"], group["correct_answer"])
    #print(f"\nContingency Table for {nback_type}:\n", contingency_table)

    # Perform chi-squared test
    chi2, p, dof, expected = chi2_contingency(contingency_table)

    # Save results
    results[nback_type] = {"chi2": chi2, "p_value": p, "dof": dof, "expected": expected}

# Display results
for nback_type, stats in results.items():
    print(f"\n{nback_type} Results:")
    print(f"Chi-Squared: {stats['chi2']:.3f}")
    print(f"P-Value: {stats['p_value']:.3f}")
    print(f"Degrees of Freedom: {stats['dof']}")
    #print(f"Expected Frequencies:\n{stats['expected']}")

# Correlations between coding and nback
correlaton of being exactly same linear vs growing in the same direction.

In [ ]:
# Filter n-back data to match participants in FITB
aligned_nback = nback_data[
    (nback_data["TMS_Type"] == "CM") & (nback_data["nback_type"] == "2-back")
]
aligned_fitb = FITB_data[
    (FITB_data["TMS_Type"] == "CM") & (FITB_data["identifer_no"] == 2)
]

# Merge datasets based on a common key (e.g., 'participant_id')
merged_data = pd.merge(aligned_nback, aligned_fitb, on="participant_id")


corr, p_value = pearsonr(merged_data["response time"], merged_data["duration_y"]) #pearson correlation assumes normal distribution
print("Correlation:", corr)
print("P-Value:", p_value)